# LLM Cluster Features - OLLAMA Reproducibility

Ноутбук строит LLM-признаки, сравнивает фиксированные кластерные сценарии и сохраняет артефакты локально.


In [ ]:
import os, sys, subprocess, re, json, time, hashlib, random, shutil, urllib.request, shlex
from pathlib import Path

PROJECT_DIR = Path(os.environ.get("OLLAMA_PROJECT_DIR", "llm_local_runs/ollama_local_pristine")).expanduser()
INPUT_DIR = Path(os.environ.get("LLM_INPUT_DIR", PROJECT_DIR / "inputs")).expanduser()
LLM_CACHE_DIR = Path(os.environ.get("OLLAMA_LLM_CACHE_DIR", PROJECT_DIR / "llm_feature_cache_ollama_local_pristine")).expanduser()
OUT_DIR = Path(os.environ.get("OLLAMA_REPRO_OUT_DIR", PROJECT_DIR / "artifacts_reproducibility_ollama_5runs")).expanduser()
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

OLLAMA_MODEL = os.getenv('OLLAMA_MODEL', 'llama3.2:3b')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://127.0.0.1:11434')
OLLAMA_CHAT_URL = OLLAMA_BASE_URL + '/api/chat'
OLLAMA_TIMEOUT = 180
OLLAMA_MAX_RETRIES = 3
OLLAMA_NUM_PREDICT = 256

os.environ['OLLAMA_MODELS'] = str(PROJECT_DIR / 'ollama_models')
Path(os.environ['OLLAMA_MODELS']).mkdir(parents=True, exist_ok=True)

def _run(cmd, shell=False, check=True):
    subprocess.run(cmd, shell=shell, check=check)

def _find_ollama_asset():
    req = urllib.request.Request(
        'https://api.github.com/repos/ollama/ollama/releases/latest',
        headers={'User-Agent': 'local'},
    )
    with urllib.request.urlopen(req, timeout=30) as r:
        data = json.loads(r.read().decode())

    assets = data.get('assets', [])

    candidates = []
    for a in assets:
        name = a['name']
        url = a['browser_download_url']

        # нужен обычный linux amd64, не rocm / не arm / не jetpack
        if 'linux-amd64' not in name:
            continue
        if any(x in name.lower() for x in ['rocm', 'arm64', 'jetpack']):
            continue

        # предпочитаем tar.zst, но поддержим и старые форматы
        if name.endswith('.tar.zst'):
            prio = 0
        elif name.endswith('.tgz') or name.endswith('.tar.gz'):
            prio = 1
        else:
            continue

        candidates.append((prio, name, url))

    if not candidates:
        raise RuntimeError(
            f"No suitable linux-amd64 archive in release; "
            f"assets={[a['name'] for a in assets]}"
        )

    candidates.sort(key=lambda x: x[0])
    _, asset_name, asset_url = candidates[0]
    return asset_name, asset_url

def _extract_ollama_archive(local_path, asset_name):
    # Для .tar.zst нужен zstd
    if asset_name.endswith('.tar.zst'):
        _run("command -v zstd >/dev/null 2>&1 || (apt-get update -y && apt-get install -y zstd)", shell=True, check=False)
        _run(f"tar --zstd -xf {shlex.quote(local_path)} -C /usr/local", shell=True)
    elif asset_name.endswith('.tgz') or asset_name.endswith('.tar.gz'):
        _run(f"tar -xzf {shlex.quote(local_path)} -C /usr/local", shell=True)
    else:
        raise RuntimeError(f"Unsupported archive format: {asset_name}")

# Ставим ollama только если не установлен
if shutil.which('ollama') is None and not Path('/usr/local/bin/ollama').exists():
    asset_name, asset_url = _find_ollama_asset()
    local_path = f'/tmp/{asset_name}'
    print('Ollama asset:', asset_name, asset_url)
    urllib.request.urlretrieve(asset_url, local_path)
    _extract_ollama_archive(local_path, asset_name)

os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH', '')
OLLAMA_BIN = shutil.which('ollama') or '/usr/local/bin/ollama'

_run([OLLAMA_BIN, '-v'], check=False)

# Поднимаем сервер
_run("pkill -f 'ollama serve' || true", shell=True, check=False)
_run(f"nohup {shlex.quote(OLLAMA_BIN)} serve > /tmp/ollama.log 2>&1 &", shell=True)
time.sleep(3)

import requests

def _ollama_alive():
    try:
        return requests.get(OLLAMA_BASE_URL + '/api/tags', timeout=3).status_code == 200
    except Exception:
        return False

t0 = time.time()
while time.time() - t0 < 90 and not _ollama_alive():
    time.sleep(1)

if not _ollama_alive():
    print('---- /tmp/ollama.log ----')
    try:
        print(Path('/tmp/ollama.log').read_text())
    except Exception:
        pass
    raise RuntimeError('Ollama server failed to start')

_run([OLLAMA_BIN, 'pull', OLLAMA_MODEL])

import numpy as np
import pandas as pd

DATA_PATH = INPUT_DIR / 'data_finall_without_TTM.csv'
if not DATA_PATH.exists():
    alt = PROJECT_DIR / 'data_finall_without_TTM.csv'
    assert alt.exists(), f'data not found: {DATA_PATH} / {alt}'
    DATA_PATH = alt

assert LLM_CACHE_DIR.exists(), LLM_CACHE_DIR

BACKEND_TAG = 'ollama_' + re.sub(r'[^a-zA-Z0-9_]+', '_', OLLAMA_MODEL)
target_col = 'duration_hours'
cap = 960

# Сколько строк перепредсказать заново (None = все, что есть в кэше).
SAMPLE_SIZE = 200
N_REPEATS = 5
SAMPLE_SEED = 42

print('drive project:', PROJECT_DIR)
print('cache dir:', LLM_CACHE_DIR)
print('files in cache:', len(list(LLM_CACHE_DIR.glob('*.json'))))
print('out dir:', OUT_DIR)
print('backend tag:', BACKEND_TAG)

In [ ]:
df = pd.read_csv(DATA_PATH)
split = int(len(df) * 0.8)
train_full = df.iloc[:split].copy()
test_full  = df.iloc[split:].copy()
train_filt = train_full[train_full[target_col] < cap].copy()

llm_train_raw = train_filt.reset_index(drop=True).copy()
llm_test_raw  = test_full.reset_index(drop=True).copy()
print('train:', llm_train_raw.shape, 'test:', llm_test_raw.shape)

In [ ]:
# === Копия логики построения payload/prompt из Llm_cluster_features_ollama.ipynb ===
LLM_EXCLUDE_COLS = {
    'duration_hours','Ключ','Задача','Статус','Резолюция','Создано','Дата завершения','created_dt',
    'Бэклог','Заблокирован','В работе','Раскатка','Merged','Протестировано','Ревью',
    'Можно тестировать','Тестируется',
}
CAT_PREFIXES = ['Приоритет_','Тип_','Команда_']

def is_binary_series(s):
    vals = set(pd.Series(s.dropna().unique()).tolist())
    return vals.issubset({0,1,0.0,1.0,False,True})

def infer_llm_feature_groups(df_part):
    safe = [c for c in df_part.columns if c not in LLM_EXCLUDE_COLS]
    binary = [c for c in safe if is_binary_series(df_part[c])]
    numeric = [c for c in safe if c not in binary]
    cat_groups, used = {}, set()
    for pref in CAT_PREFIXES:
        cols = [c for c in binary if c.startswith(pref)]
        if cols:
            cat_groups[pref] = cols
            used.update(cols)
    tags = [c for c in binary if c not in used]
    return {'numeric_cols': numeric, 'cat_groups': cat_groups, 'tag_cols': tags}

feature_groups = infer_llm_feature_groups(llm_train_raw)

def _to_jsonable(v):
    if pd.isna(v): return None
    if isinstance(v, np.integer): return int(v)
    if isinstance(v, np.floating): return float(v)
    if isinstance(v, np.bool_): return bool(v)
    if isinstance(v, pd.Timestamp): return v.isoformat()
    return v

def compact_task_payload(row, groups, max_tags=20):
    out = {'numeric_features': {}, 'priority': None, 'task_type': None, 'team': None, 'active_tags': []}
    for c in groups['numeric_cols']:
        out['numeric_features'][c] = _to_jsonable(row[c])
    for pref, cols in groups['cat_groups'].items():
        active = [c[len(pref):] for c in cols if row[c] == 1]
        val = active[0] if active else None
        if pref == 'Приоритет_': out['priority'] = val
        elif pref == 'Тип_':      out['task_type'] = val
        elif pref == 'Команда_':  out['team'] = val
    out['active_tags'] = [c for c in groups['tag_cols'] if row[c] == 1][:max_tags]
    return out

LLM_FEATURE_SYSTEM_PROMPT = """
Ты аналитик процессов разработки ПО.
По метаданным задачи верни компактный набор эвристических признаков.
Не придумывай новые факты и не используй скрытые знания о будущем.
Если информации недостаточно, выбирай нейтральные значения.
Верни строго JSON.
"""

LLM_FEATURE_KEYS = [
    'execution_complexity_1_5','coordination_risk_1_5','testing_effort_1_5',
    'uncertainty_1_5','urgent_delivery_0_1','likely_long_task_0_1',
]

def make_llm_feature_prompt(task_payload, cluster_context=None):
    payload = {'task_metadata': task_payload, 'cluster_context': cluster_context}
    return f"""
Верни JSON следующего формата:
{{
  \"execution_complexity_1_5\": int,
  \"coordination_risk_1_5\": int,
  \"testing_effort_1_5\": int,
  \"uncertainty_1_5\": int,
  \"urgent_delivery_0_1\": int,
  \"likely_long_task_0_1\": int
}}

Данные:
{json.dumps(payload, ensure_ascii=False, indent=2)}
"""

In [ ]:
def _clean_json_text(text):
    if not text: return ''
    text = text.strip()
    text = re.sub(r'^```(?:json)?\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s*```$', '', text)
    m = re.search(r'\{.*\}', text, flags=re.DOTALL)
    return m.group(0).strip() if m else text

def call_ollama_json(system_prompt, user_prompt):
    last_err = None
    for attempt in range(OLLAMA_MAX_RETRIES):
        try:
            prompt = user_prompt if attempt == 0 else user_prompt + '\n\nReturn valid JSON only.'
            payload = {
                'model': OLLAMA_MODEL,
                'stream': False,
                'messages': [
                    {'role':'system','content':system_prompt},
                    {'role':'user','content':prompt},
                ],
                'options': {'temperature': 0, 'num_predict': OLLAMA_NUM_PREDICT},
            }
            r = requests.post(OLLAMA_CHAT_URL, json=payload, timeout=OLLAMA_TIMEOUT)
            r.raise_for_status()
            raw_text = r.json().get('message', {}).get('content', '')
            data = json.loads(_clean_json_text(raw_text))
            return {k: data.get(k, np.nan) for k in LLM_FEATURE_KEYS} | {'parse_error': 0}
        except Exception as e:
            last_err = e
            time.sleep(2 ** attempt)
    return {k: np.nan for k in LLM_FEATURE_KEYS} | {'parse_error': 1, 'raw_error': str(last_err)}

def cache_key_for(system_prompt, user_prompt):
    return hashlib.sha256((BACKEND_TAG + system_prompt + user_prompt).encode('utf-8')).hexdigest()

def load_cached(system_prompt, user_prompt):
    p = LLM_CACHE_DIR / f'{cache_key_for(system_prompt, user_prompt)}.json'
    if not p.exists(): return None
    return json.loads(p.read_text(encoding='utf-8'))

In [ ]:
def iter_rows(df_part, split_name):
    for idx, r in df_part.iterrows():
        payload = compact_task_payload(r, feature_groups, max_tags=20)
        user_prompt = make_llm_feature_prompt(payload, cluster_context=None)
        yield split_name, int(idx), LLM_FEATURE_SYSTEM_PROMPT, user_prompt

all_items = list(iter_rows(llm_train_raw, 'train')) + list(iter_rows(llm_test_raw, 'test'))
cached_items = [it for it in all_items if (LLM_CACHE_DIR / f'{cache_key_for(it[2], it[3])}.json').exists()]
print(f'total rows={len(all_items)}, with cache={len(cached_items)}')

rng = random.Random(SAMPLE_SEED)
if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(cached_items):
    sample = rng.sample(cached_items, SAMPLE_SIZE)
else:
    sample = cached_items
assert len(sample) == SAMPLE_SIZE, f'Expected {SAMPLE_SIZE} cached rows, got {len(sample)}'
print(f'will re-query {len(sample)} rows x {N_REPEATS} repeats = {len(sample) * N_REPEATS} fresh requests')

In [ ]:
rows = []
t0 = time.time()
total_calls = len(sample) * N_REPEATS
call_i = 0

for repeat_id in range(1, N_REPEATS + 1):
    print(f'=== repeat {repeat_id}/{N_REPEATS}: {len(sample)} rows ===', flush=True)
    for n, (split_name, row_id, sys_p, usr_p) in enumerate(sample, start=1):
        call_i += 1
        cached = load_cached(sys_p, usr_p) or {}
        fresh = call_ollama_json(sys_p, usr_p)
        rec = {'split': split_name, 'row_id': row_id, 'repeat_id': repeat_id}
        for k in LLM_FEATURE_KEYS:
            rec[f'{k}__baseline'] = cached.get(k)
            rec[f'{k}__fresh'] = fresh.get(k)
        rec['parse_error_baseline'] = int(cached.get('parse_error', 0) or 0)
        rec['parse_error_fresh'] = int(fresh.get('parse_error', 0) or 0)
        rows.append(rec)
        if call_i == 1 or call_i % 10 == 0 or n == len(sample):
            print(
                f'call {call_i}/{total_calls} | repeat={repeat_id} row={n}/{len(sample)} '
                f'elapsed={time.time()-t0:.1f}s',
                flush=True,
            )
        if call_i % 25 == 0:
            pd.DataFrame(rows).to_csv(OUT_DIR / 'reproducibility_raw_5runs.csv', index=False)

raw_df = pd.DataFrame(rows)
raw_df.to_csv(OUT_DIR / 'reproducibility_raw_5runs.csv', index=False)
print('saved:', OUT_DIR / 'reproducibility_raw_5runs.csv', raw_df.shape)
raw_df.head()


In [ ]:
# === Statistical summary helpers ===
from math import erfc, sqrt


def _wilson_ci(successes, n, z=1.959963984540054):
    if n == 0:
        return np.nan, np.nan
    phat = successes / n
    denom = 1 + z * z / n
    center = (phat + z * z / (2 * n)) / denom
    half = z * sqrt((phat * (1 - phat) + z * z / (4 * n)) / n) / denom
    return center - half, center + half


def _normal_2sided_p(z):
    if pd.isna(z):
        return np.nan
    return float(erfc(abs(float(z)) / sqrt(2)))


def _paired_mean_test(delta):
    x = pd.to_numeric(pd.Series(delta), errors='coerce').dropna()
    n = int(len(x))
    if n < 2:
        return {'mean_signed_diff': np.nan, 'std_signed_diff': np.nan, 'se_signed_diff': np.nan,
                'z_or_t': np.nan, 'p_value_mean_shift': np.nan}
    mean = float(x.mean())
    std = float(x.std(ddof=1))
    se = std / sqrt(n) if std > 0 else 0.0
    stat = mean / se if se > 0 else (0.0 if abs(mean) < 1e-12 else np.inf)
    try:
        from scipy import stats
        p = float(stats.ttest_1samp(x, 0.0, nan_policy='omit').pvalue) if std > 0 else (1.0 if abs(mean) < 1e-12 else 0.0)
    except Exception:
        p = _normal_2sided_p(stat) if np.isfinite(stat) else (1.0 if abs(mean) < 1e-12 else 0.0)
    return {'mean_signed_diff': mean, 'std_signed_diff': std, 'se_signed_diff': se,
            'z_or_t': float(stat), 'p_value_mean_shift': p}


def _feature_summary(df_part, feature, repeat_id='pooled'):
    b = pd.to_numeric(df_part[f'{feature}__baseline'], errors='coerce')
    f = pd.to_numeric(df_part[f'{feature}__fresh'], errors='coerce')
    mask = (~b.isna()) & (~f.isna())
    n = int(mask.sum())
    exact = (f[mask] == b[mask])
    successes = int(exact.sum())
    ci_low, ci_high = _wilson_ci(successes, n)
    diff = (f[mask] - b[mask]).abs()
    signed = f[mask] - b[mask]
    paired = _paired_mean_test(signed)
    return {
        'repeat_id': repeat_id,
        'feature': feature,
        'n': n,
        'exact_matches': successes,
        'exact_match_rate': float(successes / n) if n else np.nan,
        'exact_match_ci95_low': ci_low,
        'exact_match_ci95_high': ci_high,
        'mean_abs_diff': float(diff.mean()) if n else np.nan,
        'median_abs_diff': float(diff.median()) if n else np.nan,
        'max_abs_diff': float(diff.max()) if n else np.nan,
        'baseline_mean': float(b[mask].mean()) if n else np.nan,
        'fresh_mean': float(f[mask].mean()) if n else np.nan,
        'fresh_std': float(f[mask].std(ddof=1)) if n > 1 else np.nan,
        'pearson': float(np.corrcoef(b[mask], f[mask])[0, 1])
            if n > 1 and b[mask].std() > 0 and f[mask].std() > 0 else np.nan,
        **paired,
        'mean_shift_significant_0_05': bool(paired['p_value_mean_shift'] < 0.05)
            if not pd.isna(paired['p_value_mean_shift']) else False,
    }


In [ ]:
# === Per-run and pooled summaries versus baseline/cache ===
by_run_rows = []
for repeat_id, part in raw_df.groupby('repeat_id', sort=True):
    for k in LLM_FEATURE_KEYS:
        by_run_rows.append(_feature_summary(part, k, repeat_id=int(repeat_id)))

summary_by_run = pd.DataFrame(by_run_rows)
summary_by_run.to_csv(OUT_DIR / 'reproducibility_summary_by_run.csv', index=False)

pooled_rows = [_feature_summary(raw_df, k, repeat_id='pooled_5_runs') for k in LLM_FEATURE_KEYS]
summary_overall = pd.DataFrame(pooled_rows)
summary_overall.to_csv(OUT_DIR / 'reproducibility_summary_overall.csv', index=False)

print('saved:', OUT_DIR / 'reproducibility_summary_by_run.csv', summary_by_run.shape)
print('saved:', OUT_DIR / 'reproducibility_summary_overall.csv', summary_overall.shape)
summary_overall


In [ ]:
# === Row-level stability across all 5 repeats ===
row_records = []
for (split_name, row_id), part in raw_df.groupby(['split', 'row_id'], sort=False):
    rec = {'split': split_name, 'row_id': int(row_id), 'n_repeats_observed': int(part['repeat_id'].nunique())}
    rec['parse_errors_fresh_total'] = int(part['parse_error_fresh'].sum())
    feature_all_equal_baseline = []
    feature_all_runs_identical = []
    max_abs_diffs = []
    for k in LLM_FEATURE_KEYS:
        baseline_vals = pd.to_numeric(part[f'{k}__baseline'], errors='coerce').dropna().unique()
        baseline = baseline_vals[0] if len(baseline_vals) else np.nan
        fresh = pd.to_numeric(part[f'{k}__fresh'], errors='coerce')
        rec[f'{k}__baseline'] = baseline
        for repeat_id in range(1, N_REPEATS + 1):
            vals = fresh[part['repeat_id'] == repeat_id]
            rec[f'{k}__run{repeat_id}'] = vals.iloc[0] if len(vals) else np.nan
        valid = fresh.dropna()
        all_equal_baseline = bool(len(valid) == N_REPEATS and not pd.isna(baseline) and (valid == baseline).all())
        all_identical = bool(len(valid) == N_REPEATS and valid.nunique(dropna=False) == 1)
        feature_all_equal_baseline.append(all_equal_baseline)
        feature_all_runs_identical.append(all_identical)
        max_abs_diffs.append(float((valid - baseline).abs().max()) if len(valid) and not pd.isna(baseline) else np.nan)
        rec[f'{k}__all_runs_equal_baseline'] = all_equal_baseline
        rec[f'{k}__all_runs_identical'] = all_identical
        rec[f'{k}__fresh_std_across_runs'] = float(valid.std(ddof=1)) if len(valid) > 1 else np.nan
    rec['row_all_features_all_runs_equal_baseline'] = bool(all(feature_all_equal_baseline))
    rec['row_all_features_all_runs_identical'] = bool(all(feature_all_runs_identical))
    finite_diffs = [x for x in max_abs_diffs if np.isfinite(x)]
    rec['row_max_abs_diff_any_feature'] = float(max(finite_diffs)) if finite_diffs else np.nan
    row_records.append(rec)

row_stability = pd.DataFrame(row_records)
row_stability.to_csv(OUT_DIR / 'reproducibility_row_stability.csv', index=False)
print('saved:', OUT_DIR / 'reproducibility_row_stability.csv', row_stability.shape)
row_stability.head()


In [ ]:
# === Executive summary ===
all_feature_cols_baseline = [f'{k}__baseline' for k in LLM_FEATURE_KEYS]
all_feature_cols_fresh = [f'{k}__fresh' for k in LLM_FEATURE_KEYS]
run_exact_rows = []
for repeat_id, part in raw_df.groupby('repeat_id', sort=True):
    baseline_vals = part[all_feature_cols_baseline].to_numpy()
    fresh_vals = part[all_feature_cols_fresh].to_numpy()
    row_match = (baseline_vals == fresh_vals).all(axis=1)
    run_exact_rows.append({
        'repeat_id': int(repeat_id),
        'row_exact_all_features_rate': float(row_match.mean()),
        'row_exact_all_features_count': int(row_match.sum()),
        'n_rows': int(len(row_match)),
        'parse_errors_fresh': int(part['parse_error_fresh'].sum()),
    })
row_exact_by_run = pd.DataFrame(run_exact_rows)
row_exact_by_run.to_csv(OUT_DIR / 'reproducibility_row_exact_by_run.csv', index=False)

executive_summary = {
    'sample_size_rows': int(len(sample)),
    'n_repeats': int(N_REPEATS),
    'total_fresh_requests': int(len(raw_df)),
    'feature_count': int(len(LLM_FEATURE_KEYS)),
    'row_exact_all_features_mean_across_runs': float(row_exact_by_run['row_exact_all_features_rate'].mean()),
    'row_exact_all_features_min_run': float(row_exact_by_run['row_exact_all_features_rate'].min()),
    'rows_all_5_runs_equal_baseline_rate': float(row_stability['row_all_features_all_runs_equal_baseline'].mean()),
    'rows_all_5_runs_identical_rate': float(row_stability['row_all_features_all_runs_identical'].mean()),
    'fresh_parse_error_rate': float(raw_df['parse_error_fresh'].mean()),
    'outputs': {
        'raw': str(OUT_DIR / 'reproducibility_raw_5runs.csv'),
        'summary_by_run': str(OUT_DIR / 'reproducibility_summary_by_run.csv'),
        'summary_overall': str(OUT_DIR / 'reproducibility_summary_overall.csv'),
        'row_stability': str(OUT_DIR / 'reproducibility_row_stability.csv'),
        'row_exact_by_run': str(OUT_DIR / 'reproducibility_row_exact_by_run.csv'),
    },
}
with open(OUT_DIR / 'reproducibility_executive_summary.json', 'w', encoding='utf-8') as f:
    json.dump(executive_summary, f, ensure_ascii=False, indent=2)

print(json.dumps(executive_summary, ensure_ascii=False, indent=2))
print('\n--- ВЫВОД ---')
if executive_summary['rows_all_5_runs_equal_baseline_rate'] >= 0.95:
    print('Провайдер воспроизводит baseline стабильно: >=95% строк полностью совпали с baseline во всех 5 fresh-прогонах.')
elif executive_summary['row_exact_all_features_mean_across_runs'] >= 0.70:
    print('Есть частичная воспроизводимость: одиночные прогоны часто совпадают, но 5/5 по строкам ниже строгого порога.')
else:
    print('Воспроизводимость низкая: результаты заметно шумят относительно baseline/cache.')

print('\nДля статистической интерпретации смотри exact_match_ci95_low/high и p_value_mean_shift в summary_overall.')


In [ ]:
# === Correlation matrices: baseline vs 5 fresh runs ===
import matplotlib.pyplot as plt

CORR_DIR = OUT_DIR / "correlation_matrices"
CORR_DIR.mkdir(parents=True, exist_ok=True)

corr_records = []

for feature in LLM_FEATURE_KEYS:
    wide_rows = []

    for (split_name, row_id), part in raw_df.groupby(["split", "row_id"], sort=False):
        rec = {"split": split_name, "row_id": row_id}

        baseline_vals = pd.to_numeric(part[f"{feature}__baseline"], errors="coerce").dropna().unique()
        rec["baseline"] = baseline_vals[0] if len(baseline_vals) else np.nan

        for repeat_id in range(1, N_REPEATS + 1):
            vals = part.loc[part["repeat_id"] == repeat_id, f"{feature}__fresh"]
            rec[f"run_{repeat_id}"] = pd.to_numeric(vals, errors="coerce").iloc[0] if len(vals) else np.nan

        wide_rows.append(rec)

    wide_df = pd.DataFrame(wide_rows)
    corr_cols = ["baseline"] + [f"run_{i}" for i in range(1, N_REPEATS + 1)]
    corr = wide_df[corr_cols].corr(method="pearson")

    corr.to_csv(CORR_DIR / f"{feature}_correlation_matrix.csv")

    plt.figure(figsize=(7, 6))
    im = plt.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
    plt.colorbar(im, fraction=0.046, pad=0.04)

    plt.xticks(range(len(corr_cols)), corr_cols, rotation=45, ha="right")
    plt.yticks(range(len(corr_cols)), corr_cols)
    plt.title(f"Correlation matrix: {feature}")

    for i in range(len(corr_cols)):
        for j in range(len(corr_cols)):
            val = corr.iloc[i, j]
            label = "" if pd.isna(val) else f"{val:.2f}"
            plt.text(j, i, label, ha="center", va="center", color="black", fontsize=9)

    plt.tight_layout()
    png_path = CORR_DIR / f"{feature}_correlation_matrix.png"
    plt.savefig(png_path, dpi=200, bbox_inches="tight")
    plt.show()

    corr_long = corr.reset_index().melt(
        id_vars="index",
        var_name="col_b",
        value_name="pearson_corr",
    ).rename(columns={"index": "col_a"})
    corr_long.insert(0, "feature", feature)
    corr_records.append(corr_long)

all_corr = pd.concat(corr_records, ignore_index=True)
all_corr.to_csv(OUT_DIR / "correlation_matrices_all_features_long.csv", index=False)

print("Saved correlation matrices to:", CORR_DIR)
print("Saved long summary to:", OUT_DIR / "correlation_matrices_all_features_long.csv")


In [ ]:
# === Compact row of correlation matrices with one shared color legend ===
import matplotlib.pyplot as plt

CORR_DIR = OUT_DIR / "correlation_matrices"
CORR_DIR.mkdir(parents=True, exist_ok=True)

model_name = (
    globals().get("OPENAI_MODEL_NAME")

    or globals().get("CLAUDE_MODEL")
    or globals().get("OLLAMA_MODEL")
    or globals().get("QWEN_MODEL_ID")
    or "LLM provider"
)

corr_cols = ["baseline"] + [f"run_{i}" for i in range(1, N_REPEATS + 1)]
n_features = len(LLM_FEATURE_KEYS)

fig, axes = plt.subplots(
    1,
    n_features,
    figsize=(2.2 * n_features + 1.0, 2.5),
    constrained_layout=True,
)

if n_features == 1:
    axes = [axes]

last_im = None

for ax, feature in zip(axes, LLM_FEATURE_KEYS):
    wide_rows = []

    for (split_name, row_id), part in raw_df.groupby(["split", "row_id"], sort=False):
        rec = {"split": split_name, "row_id": row_id}

        baseline_vals = pd.to_numeric(part[f"{feature}__baseline"], errors="coerce").dropna().unique()
        rec["baseline"] = baseline_vals[0] if len(baseline_vals) else np.nan

        for repeat_id in range(1, N_REPEATS + 1):
            vals = part.loc[part["repeat_id"] == repeat_id, f"{feature}__fresh"]
            rec[f"run_{repeat_id}"] = pd.to_numeric(vals, errors="coerce").iloc[0] if len(vals) else np.nan

        wide_rows.append(rec)

    wide_df = pd.DataFrame(wide_rows)
    corr = wide_df[corr_cols].corr(method="pearson")

    last_im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_title(feature, fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_visible(False)

fig.suptitle(f"Correlation matrices for {model_name}", fontsize=12)

cbar = fig.colorbar(
    last_im,
    ax=axes,
    location="right",
    fraction=0.025,
    pad=0.02,
)
cbar.set_label("Pearson correlation", fontsize=9)

fig_path = CORR_DIR / "all_features_correlation_matrices_row_with_legend.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", fig_path)
